# A/B Test Analysis: Marketing Campaign Effectiveness

**Problem Statement:**
The marketing team ran an A/B test where the treatment group saw a brand advertisement (`ad`) and the control group saw a public service announcement (`psa`).

**Objective:**
Determine if the brand advertisement significantly increases the conversion rate compared to the PSA. We will evaluate not just if the result is *statistically* significant, but whether the effect size is *practically* significant for the business, and verify if the test was adequately powered.

In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from statsmodels.stats.power import zt_ind_solve_power
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")

## 1. Data Loading & Sanity Checks
Before running any tests, we must ensure the data is clean, verify there are no duplicate users, and check if the randomization worked (Sample Ratio Mismatch).

In [4]:
# Load dataset
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/datasets/marketing_AB.csv')
display(df.head())

# Basic info
print(f"Total rows: {df.shape[0]}")
print(f"Unique users: {df['user id'].nunique()}")
print("Missing values:\n", df.isnull().sum())

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


Total rows: 588101
Unique users: 588101
Missing values:
 Unnamed: 0       0
user id          0
test group       0
converted        0
total ads        0
most ads day     0
most ads hour    0
dtype: int64


## 2. Exploratory Data Analysis & Randomization Check
A crucial assumption of A/B testing is that the treatment and control groups are comparable. If one group has drastically different user behaviors (e.g., active on different days), the randomization may have failed.

In [5]:
# Check group sizes and conversion rates
summary = df.groupby('test group').agg(
    users=('user id', 'count'),
    conversions=('converted', 'sum')
).reset_index()

summary['conversion_rate'] = summary['conversions'] / summary['users']
display(summary)

# Formal SRM Test (Chi-Square Goodness of Fit)
total_users = df.shape[0]
planned_ad_prop = 0.96 # Assuming the test was designed for a 96/4 split
planned_psa_prop = 0.04

expected_counts = [total_users * planned_ad_prop, total_users * planned_psa_prop]
observed_counts = [summary.loc[summary['test group'] == 'ad', 'users'].values[0],
                   summary.loc[summary['test group'] == 'psa', 'users'].values[0]]

srm_stat, srm_pval = stats.chisquare(f_obs=observed_counts, f_exp=expected_counts)

print(f"SRM Chi-Square Statistic: {srm_stat:.4f}")
print(f"SRM P-value: {srm_pval:.4e}")
if srm_pval < 0.05:
    print("WARNING: Statistically significant Sample Ratio Mismatch detected. Investigate randomization logging.")
else:
    print("PASS: No significant Sample Ratio Mismatch detected based on the assumed 96/4 split.")

,test group,users,conversions,conversion_rate
0,ad,564577,14423,0.025547
1,psa,23524,420,0.017854


SRM Chi-Square Statistic: 0.0000
SRM P-value: 9.9979e-01
PASS: No significant Sample Ratio Mismatch detected based on the assumed 96/4 split.


## 3. Formal Statistical Inference: Two-Proportion Z-Test

We are comparing two independent proportions.

**Hypotheses:**
*   **H0 (Null Hypothesis):** $p_{ad} - p_{psa} = 0$ (There is no difference in conversion rates)
*   **H1 (Alternative Hypothesis):** $p_{ad} - p_{psa} \neq 0$ (There is a difference)

**Assumptions Check:**
1.  **Independence:** Users are randomly assigned and independent (assumed true based on test design).
2.  **Sample Size:** Both $np$ and $n(1-p)$ are > 10 for both groups. With 500k+ rows, we easily meet this assumption for the Normal approximation.

We will use a standard alpha level of 0.05.

In [6]:
# Extract counts
ad_conversions = summary.loc[summary['test group'] == 'ad', 'conversions'].values[0]
psa_conversions = summary.loc[summary['test group'] == 'psa', 'conversions'].values[0]

ad_total = summary.loc[summary['test group'] == 'ad', 'users'].values[0]
psa_total = summary.loc[summary['test group'] == 'psa', 'users'].values[0]

# Perform Z-test
count = np.array([ad_conversions, psa_conversions])
nobs = np.array([ad_total, psa_total])

z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')

# Calculate 95% Confidence Interval for the DIFFERENCE in proportions
# statsmodels confint_proportions_2indep uses Newcombe's method by default
ci_lower, ci_upper = confint_proportions_2indep(ad_conversions, ad_total, psa_conversions, psa_total, alpha=0.05)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4e}")
print(f"95% CI for the absolute difference in conversion rate: [{ci_lower:.4f}, {ci_upper:.4f}]")

Z-statistic: 7.3701
P-value: 1.7053e-13
95% CI for the absolute difference in conversion rate: [0.0059, 0.0094]


## 4. A Priori Power Analysis
Calculating power *after* observing the result (post-hoc power) is a statistical fallacy, as it is mathematically just a transformation of the p-value.

Instead, we validate the test design: Assuming a baseline conversion rate of ~1.78% (the PSA group), if we wanted to detect a **10% relative lift** (a Minimum Detectable Effect of ~0.178% absolute), did we have enough samples to achieve 80% power at our actual allocation ratio?

In [7]:
baseline_rate = psa_conversions / psa_total
mde_relative = 0.10
mde_absolute = baseline_rate * mde_relative
expected_treatment_rate = baseline_rate + mde_absolute

effect_size_mde = sm.stats.proportion_effectsize(expected_treatment_rate, baseline_rate)

required_n_psa = zt_ind_solve_power(
    effect_size=effect_size_mde,
    nobs1=None,
    alpha=0.05,
    power=0.80,
    ratio=(ad_total / psa_total),
    alternative='two-sided'
)

print(f"Baseline Rate: {baseline_rate:.4%}")
print(f"Targeting a 10% relative lift (MDE: {mde_absolute:.4%})")
print(f"Required sample size (PSA group) for 80% power: {int(np.ceil(required_n_psa)):,}")
print(f"Actual sample size (PSA group): {psa_total:,}")

if psa_total >= required_n_psa:
    print("Verdict: The test was adequately powered to detect the MDE.")
else:
    print("Verdict: The test was UNDERPOWERED for this specific MDE.")

Baseline Rate: 1.7854%
Targeting a 10% relative lift (MDE: 0.1785%)
Required sample size (PSA group) for 80% power: 47,155
Actual sample size (PSA group): 23,524
Verdict: The test was UNDERPOWERED for this specific MDE.


## 5. Statistical vs. Practical Significance

**Statistical Conclusion:**
Since our p-value is $1.7053 \times 10^{-13}$[cite: 1], we reject the null hypothesis. The difference in conversion rates is highly statistically significant.

**Practical (Business) Significance:**
While the result is statistically significant, we must look at the Confidence Interval to determine business value.
*   The baseline conversion rate (PSA) is $1.79\%$.
*   The Ad conversion rate is $2.55\%$.
*   The absolute lift is $\sim 0.77\%$ (The 95% CI shows it ranges between $0.59\%$ and $0.94\%$[cite: 1]).
*   The relative lift is roughly $43\%$.

*Business Verdict:* If the cost of serving the Ad instead of the PSA is lower than the revenue generated by an additional ~7 to 9 conversions per 1,000 users, this campaign is a success.

## 6. Independent Replication: Bootstrap Confidence Interval
To independently validate the analytic Newcombe Confidence Interval, we will use a parametric bootstrap to simulate 10,000 empirical trials and build a 95% CI on the difference in proportions.

In [8]:
np.random.seed(42)
n_bootstraps = 10000

# Parametric bootstrap
boot_ad_conversions = np.random.binomial(ad_total, ad_conversions/ad_total, n_bootstraps)
boot_psa_conversions = np.random.binomial(psa_total, psa_conversions/psa_total, n_bootstraps)

boot_ad_rates = boot_ad_conversions / ad_total
boot_psa_rates = boot_psa_conversions / psa_total
boot_diffs = boot_ad_rates - boot_psa_rates

boot_ci_lower, boot_ci_upper = np.percentile(boot_diffs, [2.5, 97.5])

print(f"Analytic 95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"Bootstrap 95% CI: [{boot_ci_lower:.4f}, {boot_ci_upper:.4f}]")

Analytic 95% CI: [0.0059, 0.0094]
Bootstrap 95% CI: [0.0059, 0.0094]


## 7. (Optional) Segment Analysis & Multiple Comparisons
Does the ad perform better on certain days of the week? Because we test multiple days, we increase our chance of a False Positive (Type I error). We apply a **Bonferroni Correction** to our alpha threshold.

In [9]:
days = df['most ads day'].unique()
alpha = 0.05
bonferroni_alpha = alpha / len(days)

print(f"Original Alpha: {alpha}")
print(f"Bonferroni Corrected Alpha: {bonferroni_alpha:.4f}\n")

for day in days:
    subset = df[df['most ads day'] == day]

    ad_conv = subset[(subset['test group'] == 'ad') & (subset['converted'] == True)].shape[0]
    ad_tot = subset[subset['test group'] == 'ad'].shape[0]

    psa_conv = subset[(subset['test group'] == 'psa') & (subset['converted'] == True)].shape[0]
    psa_tot = subset[subset['test group'] == 'psa'].shape[0]

    stat, pval = proportions_ztest([ad_conv, psa_conv], [ad_tot, psa_tot])

    significant = pval < bonferroni_alpha
    print(f"Day: {day:10} | P-value: {pval:.4e} | Significant: {significant}")

Original Alpha: 0.05
Bonferroni Corrected Alpha: 0.0071

Day: Monday     | P-value: 5.0782e-04 | Significant: True
Day: Tuesday    | P-value: 6.6344e-07 | Significant: True
Day: Friday     | P-value: 1.1569e-02 | Significant: False
Day: Saturday   | P-value: 7.4838e-03 | Significant: False
Day: Wednesday  | P-value: 3.7641e-04 | Significant: True
Day: Sunday     | P-value: 1.5719e-01 | Significant: False
Day: Thursday   | P-value: 5.5475e-01 | Significant: False


## 8. Limitations & Internal Validity
*   **Non-Strict Randomization:** In pure A/B testing, exposure is strictly randomized. In this marketing dataset, exposure relies heavily on user behavior (e.g., how often they browse to trigger an ad). Users who see 300 ads are inherently different from users who see 1 ad, meaning ad frequency acts as a confounding variable rather than a pure treatment application.
*   **Novelty Effect Unmeasured:** We do not have longitudinal data to track if the lift sustains over time or if users simply reacted to seeing a new creative format temporarily.